# Q6: Outlier Detection (Z-score and IQR) - BSDS500 Image Statistics

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

BASE = r"C:\Users\cqds\Downloads\bsds500archive"
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")
IMAGES_TRAIN = os.path.join(BASE, "images", "train")

### Compute one summary-statistics row per image

In [ ]:
def image_summary_features(images_dir):
    filenames = sorted(f for f in os.listdir(images_dir) if f.lower().endswith(IMAGE_EXTENSIONS))
    rows = []
    for fname in filenames:
        img = np.array(Image.open(os.path.join(images_dir, fname)).convert("RGB"), dtype=float) / 255.0
        gray = img.mean(axis=2)
        grad_y = np.abs(np.diff(gray, axis=0, prepend=gray[:1, :]))
        grad_x = np.abs(np.diff(gray, axis=1, prepend=gray[:, :1]))
        grad_mag = np.sqrt(grad_x ** 2 + grad_y ** 2)
        rows.append({
            "filename": fname,
            "mean_R": img[:, :, 0].mean(), "mean_G": img[:, :, 1].mean(), "mean_B": img[:, :, 2].mean(),
            "mean_gray": gray.mean(), "std_gray": gray.std(),
            "mean_gradient": grad_mag.mean(),
        })
    return pd.DataFrame(rows)

df = image_summary_features(IMAGES_TRAIN)
print(df.head())
print("Shape:", df.shape)

### Basic statistics

In [ ]:
numeric_cols = ["mean_R", "mean_G", "mean_B", "mean_gray", "std_gray", "mean_gradient"]
print(df[numeric_cols].describe())

### Z-score method on mean_gray

In [ ]:
col = df["mean_gray"]
col_mean, col_std = col.mean(), col.std()
z_scores = (col - col_mean) / col_std
z_outliers = df[np.abs(z_scores) > 3]
print("Z-score outliers in mean_gray:", len(z_outliers))
print(z_outliers[["filename", "mean_gray"]])

### IQR method on mean_gray

In [ ]:
q1 = col.quantile(0.25)
q3 = col.quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
iqr_outliers = df[(col < lower_fence) | (col > upper_fence)]
print("Q1:", round(q1, 4), " Q3:", round(q3, 4), " IQR:", round(iqr, 4))
print("Lower fence:", round(lower_fence, 4), " Upper fence:", round(upper_fence, 4))
print("IQR outliers in mean_gray:", len(iqr_outliers))
print(iqr_outliers[["filename", "mean_gray"]])

### Repeat both methods on mean_gradient (texture/edge density)

In [ ]:
grad_col = df["mean_gradient"]
grad_mean, grad_std = grad_col.mean(), grad_col.std()
grad_z = (grad_col - grad_mean) / grad_std
z_outliers_grad = df[np.abs(grad_z) > 3]

q1_g, q3_g = grad_col.quantile(0.25), grad_col.quantile(0.75)
iqr_g = q3_g - q1_g
lower_g = q1_g - 1.5 * iqr_g
upper_g = q3_g + 1.5 * iqr_g
iqr_outliers_grad = df[(grad_col < lower_g) | (grad_col > upper_g)]

print("Z-score outliers in mean_gradient:", len(z_outliers_grad))
print("IQR outliers in mean_gradient:", len(iqr_outliers_grad))

### Treat outliers by capping (winsorizing)

In [ ]:
mean_gray_capped = col.clip(lower=lower_fence, upper=upper_fence)
mean_gradient_capped = grad_col.clip(lower=lower_g, upper=upper_g)
print("mean_gray std before capping:", round(col.std(), 4), " after:", round(mean_gray_capped.std(), 4))
print("mean_gradient std before capping:", round(grad_col.std(), 4), " after:", round(mean_gradient_capped.std(), 4))

### Visualize before/after with boxplots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].boxplot(col)
axes[0, 0].set_title("Mean image brightness - before treatment")
axes[0, 1].boxplot(mean_gray_capped)
axes[0, 1].set_title("Mean image brightness - after IQR capping")

axes[1, 0].boxplot(grad_col)
axes[1, 0].set_title("Mean gradient magnitude - before treatment")
axes[1, 1].boxplot(mean_gradient_capped)
axes[1, 1].set_title("Mean gradient magnitude - after IQR capping")

for ax in axes.ravel():
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()